# Falcon-7B Instruct: complete local inference

Install torch 2.10.0, transformers 4.57.6, bitsandbytes 0.50.2, accelerate 1.12.0, huggingface-hub 0.36.2 and safetensors 0.8.0. Run all cells in order. This downloads the full official ~14.4 GB checkpoint, checks both hashes and quantizes linear weights locally to NF4. Leave adequate disk/RAM headroom. CPU float16 storage for unquantized weights and float32 quantized-linear compute keep the example portable on the tested Apple host; no xformers/CUDA-only dependency or remote model code is needed. The exact original prompt, top-k sampling and 200-total-token ceiling are retained. No paid API, external inference service, account or upload. All five original source blocks remain verbatim below.

In [ ]:
import os, sys, json, hashlib, platform, gc
from pathlib import Path
import torch, bitsandbytes as bnb, transformers
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, set_seed
torch.set_num_threads(4)
cache = Path(os.environ.get('TDB_MODEL_CACHE', str(Path.home()/'.cache'/'to-data-beyond-models')))
print('Python:',sys.version.split()[0],'PyTorch:',torch.__version__,'Transformers:',transformers.__version__,'bitsandbytes:',bnb.__version__)
print('Real Falcon-7B Instruct, native Transformers implementation, local CPU NF4 inference.')

In [ ]:
model_id = 'tiiuae/falcon-7b-instruct'
revision = '8782b5c5d8c9290412416618f36a133653e85285'
checkpoint = Path(snapshot_download(model_id, revision=revision, cache_dir=cache, token=False,
    allow_patterns=['config.json','generation_config.json','tokenizer.json','tokenizer_config.json',
        'special_tokens_map.json','model.safetensors.index.json','model-*.safetensors']))
expected_hashes = {
    'model-00001-of-00002.safetensors':'b4515a8e32f81ab4fef65e10891fdb188bc45c4195620d447bc3263e132d9de5',
    'model-00002-of-00002.safetensors':'d68a175bf005c2dd7fdd9956227b5691802c8d5a5b03897ed107e591017b33ea'
}
for name, expected in expected_hashes.items():
    digest=hashlib.sha256()
    with (checkpoint/name).open('rb') as stream:
        for chunk in iter(lambda:stream.read(8*1024*1024),b''):digest.update(chunk)
    assert digest.hexdigest()==expected
tokenizer=AutoTokenizer.from_pretrained(checkpoint,local_files_only=True,trust_remote_code=False)
tokenizer.pad_token=tokenizer.eos_token
print('Official model:',model_id,'Revision:',revision)
print('Both full safetensors shards match their official SHA256 values.')

In [ ]:
quantization = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32)
model = AutoModelForCausalLM.from_pretrained(checkpoint,quantization_config=quantization,
    dtype=torch.float16,device_map={'':'cpu'},low_cpu_mem_usage=True,
    attn_implementation='sdpa',local_files_only=True,trust_remote_code=False).eval()
assert model.__class__.__name__=='FalconForCausalLM'
quantized_layers=[m for m in model.modules()if isinstance(m,bnb.nn.Linear4bit)]
assert len(quantized_layers)==4*model.config.num_hidden_layers
assert all(m.weight.quant_state.quant_type=='nf4' and m.weight.quant_state.nested for m in quantized_layers)
assert all(p.device.type=='cpu'for p in model.parameters())
generator=pipeline('text-generation',model=model,tokenizer=tokenizer)
print('Native model:',model.__class__.__name__,'NF4 linear layers:',len(quantized_layers))
print('Actual model storage bytes:',model.get_memory_footprint())

In [ ]:
prompt = 'Girafatron is obsessed with giraffes, the most glorious animal on the face of this Earth. Giraftron believes all other animals are irrelevant when compared to the glorious majesty of the giraffe.\nDaniel: Hello, Girafatron!\nGirafatron:'
encoded=tokenizer(prompt,return_tensors='pt')
input_tokens=encoded['input_ids'].shape[1]
assert input_tokens<200
settings=dict(max_new_tokens=200-input_tokens,do_sample=True,top_k=10,top_p=1.0,
    temperature=1.0,repetition_penalty=1.0,num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.eos_token_id)
set_seed(21)
sequences=generator(prompt,**settings,return_full_text=True,clean_up_tokenization_spaces=False)
assert len(sequences)==1 and sequences[0]['generated_text'].startswith(prompt)
print('Result:',sequences[0]['generated_text'])

In [ ]:
# Replay the exact same sampling settings through the direct model API.
set_seed(21)
with torch.inference_mode():
    direct=model.generate(**encoded,**settings,return_dict_in_generate=True,output_scores=True)
assert 0<len(direct.scores)<=200-input_tokens
assert direct.sequences.shape[1]<=200
assert torch.equal(direct.sequences[:,:input_tokens],encoded['input_ids'])
assert all(torch.isfinite(score).sum()>=1 for score in direct.scores)
direct_text=tokenizer.decode(direct.sequences[0],skip_special_tokens=True,clean_up_tokenization_spaces=False)
assert direct_text==sequences[0]['generated_text']
assert all(p.grad is None for p in model.parameters())
print('Pipeline and direct generation are identical under the same seed and settings.')
print('Prompt tokens:',input_tokens,'Actual generated tokens:',len(direct.scores),'Total:',direct.sequences.shape[1])

In [ ]:
assert len(quantized_layers)==128 and model.config.num_hidden_layers==32
assert model.config.hidden_size==4544 and model.config.vocab_size==65024
assert not model.training and direct.sequences.dtype==torch.int64
assert torch.equal(direct.sequences[:,:input_tokens],encoded['input_ids'])
print('PASS: full official Falcon-7B weights, hash verification, actual packed NF4 layers, original prompt/budget, native local inference and direct/pipeline parity.')
del direct, sequences, generator, model, quantized_layers
gc.collect()
print('Released the inference model.')

## Original source-code appendix

Verbatim original archive examples, retained without executing their GPU-specific setup:

```python
!pip install transformers
!pip install einops
!pip install accelerate
!pip install xformers
```

```python
model = "tiiuae/falcon-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model)
```

```python
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)
```

```python
sequences = pipeline(
   "Girafatron is obsessed with giraffes, the most glorious animal on the face of this Earth. Giraftron believes all other animals are irrelevant when compared to the glorious majesty of the giraffe.\nDaniel: Hello, Girafatron!\nGirafatron:",
    max_length=200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")
```

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
import torch

model = "tiiuae/falcon-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model)
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)
sequences = pipeline(
   "Girafatron is obsessed with giraffes, the most glorious animal on the face of this Earth. Giraftron believes all other animals are irrelevant when compared to the glorious majesty of the giraffe.\nDaniel: Hello, Girafatron!\nGirafatron:",
    max_length=200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")
```